In [1]:
# ================================
# 1️⃣ Install Dependencies
# ================================
!pip install -q transformers datasets scikit-learn tqdm

# ================================
# 2️⃣ Imports
# ================================
import json
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, get_scheduler
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm
from collections import Counter

# Mixed precision
from torch.cuda.amp import autocast, GradScaler

# ================================
# 3️⃣ Device
# ================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ================================
# 4️⃣ Load Dataset
# ================================
file_path = "/content/OS_relations_fixed.json"
with open(file_path, "r") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

# ================================
# 5️⃣ Labels
# ================================
LABELS = sorted(list(set([ex["output"].strip() for ex in data])))
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

def encode_labels(example):
    example["label"] = label2id[example["output"].strip()]
    return example

dataset = dataset.map(encode_labels)

# ================================
# 6️⃣ Few-Shot Sampling (FIXED)
# ================================
N_SHOT = 20

indices = []
for label_id in range(len(LABELS)):
    label_indices = [i for i, ex in enumerate(dataset) if ex["label"] == label_id]
    random.shuffle(label_indices)  # 🔥 IMPORTANT
    indices.extend(label_indices[:N_SHOT])

# Shuffle all selected indices
random.shuffle(indices)

# Split
train_size = int(0.8 * len(indices))
train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

# ================================
# 7️⃣ Tokenizer
# ================================
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def collate_fn(batch):
    texts = [ex["input"] for ex in batch]
    labels = torch.tensor([ex["label"] for ex in batch])

    encoding = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=128,
        return_tensors="pt"
    )
    encoding["labels"] = labels
    return encoding

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=8, collate_fn=collate_fn)

# ================================
# 8️⃣ Model (FIXED pooling)
# ================================
class DistilBERTClassifier(nn.Module):
    def __init__(self, base_model, num_labels):
        super().__init__()
        self.bert = base_model
        self.classifier = nn.Linear(base_model.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # 🔥 Mean Pooling instead of CLS
        last_hidden = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        pooled = torch.sum(last_hidden * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)

        return self.classifier(pooled)

base_model = AutoModel.from_pretrained(model_name)
model = DistilBERTClassifier(base_model, len(LABELS)).to(device)

# ================================
# 9️⃣ Optimizer & Scheduler
# ================================
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 5

num_training_steps = num_epochs * len(train_loader)

scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

# 🔥 Class Weights (fix imbalance)
label_counts = Counter([dataset[i]["label"] for i in train_indices])
weights = torch.tensor([1.0 / label_counts[i] for i in range(len(LABELS))]).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

scaler = GradScaler()

# ================================
# 🔟 Training Loop
# ================================
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        with autocast():
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

    print(f"\nEpoch {epoch+1} Avg Loss: {total_loss/len(train_loader):.4f}")

# ================================
# 1️⃣1️⃣ Evaluation
# ================================
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for batch in tqdm(val_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

print("\n===== FEW-SHOT RESULTS (DistilBERT) =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

# ================================
# 🔍 Debug Predictions
# ================================
print("\nSample Predictions:")
for i in range(min(10, len(y_pred))):
    print(f"True: {id2label[y_true[i]]} | Pred: {id2label[y_pred[i]]}")

# ================================
# 1️⃣2️⃣ Save Predictions
# ================================
results = [
    {
        "input": dataset[val_indices[i]]["input"],
        "ground_truth": id2label[y_true[i]],
        "predicted": id2label[y_pred[i]]
    }
    for i in range(len(y_pred))
]

output_file = "/content/distilbert_fewshot_predictions.json"
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\n✅ Saved predictions to {output_file}")

from google.colab import files
files.download(output_file)

Using device: cuda


Map:   0%|          | 0/1957 [00:00<?, ? examples/s]

Train size: 96, Val size: 24


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...


Epoch 5 Avg Loss: 1.5552


100%|██████████| 3/3 [00:00<00:00, 43.33it/s]



===== FEW-SHOT RESULTS (DistilBERT) =====
Accuracy : 0.3333
Precision: 0.5111
Recall   : 0.3333
F1 Score : 0.3756

Sample Predictions:
True: improves | Pred: improves
True: part_of | Pred: part_of
True: implements | Pred: based_on
True: improves | Pred: part_of
True: used_for | Pred: used_for
True: implements | Pred: based_on
True: part_of | Pred: part_of
True: no_relation | Pred: implements
True: implements | Pred: part_of
True: used_for | Pred: used_for

✅ Saved predictions to /content/distilbert_fewshot_predictions.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>